# Notebook 05 — 복합 개선 실험 (A/B/C/D 비교)

## 실험 목표
Notebook 03 QLoRA baseline 대비 세 가지 개선 방향을 독립·조합 실험한다.

| ID | 이름 | 변경점 | 예상 효과 |
|----|------|---------|-----------|
| baseline | rank16 (기존) | rank=16, aug=없음, CE | 참조값 |
| A | rank32 | LoRA rank 16 → 32 | 표현력 ↑ |
| B | augmented | albumentations 증강 | 일반화 ↑ |
| C | improved | 레이블스무딩 0.1 + 얼리스토핑 | 과적합 ↓ |
| D | best_combo | A + B + C 통합 | 종합 최적 |

> **시간**: RTX 4080 Super 기준 실험당 약 2~3시간.  
> `EXPERIMENTS_TO_RUN` 셀에서 원하는 ID만 선택해 실행 가능.

In [ ]:
%pip install -q matplotlib numpy pandas Pillow scikit-learn tqdm transformers peft accelerate bitsandbytes albumentations

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import json, re, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import albumentations as A

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import bitsandbytes as bnb

# ── 경로 설정 ─────────────────────────────────────────────
_cwd = Path.cwd()
ROOT           = _cwd if (_cwd / "data").exists() else _cwd.parent
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR    = ROOT / "data" / "results"
EXP_CKPT_DIR   = ROOT / "models" / "checkpoints" / "experiments"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXP_CKPT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DEFECT_CLASSES = [
    "crazing", "inclusion", "patches",
    "pitted_surface", "rolled-in_scale", "scratches"
]
SYSTEM_PROMPT = (
    "당신은 금속 제품 표면 불량을 분석하는 전문 AI입니다. "
    "주어진 이미지를 분석하여 불량 유형을 정확히 판단하고 "
    "반드시 JSON 형식으로만 답변하세요."
)
_FMT = '{"type": "...", "type_ko": "...", "severity": "low|medium|high", "description": "..."}'
INFERENCE_PROMPT = (
    "이 금속 표면 이미지를 분석하고 불량 정보를 JSON 형식으로 출력해줘.\n"
    "불량 유형은 반드시 다음 중 하나여야 해: "
    "crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches\n"
    f"출력 형식: {_FMT}"
)

for f in ("train.json", "val.json", "test.json"):
    assert (DATA_PROCESSED / f).exists(), f"{f} 없음 — 01_dataset.ipynb 먼저 실행"

print(f"ROOT : {ROOT}")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Notebook 03/04 결과를 baseline으로 로드
_bp = RESULTS_DIR / "finetuned_metrics.json"
BASELINE_METRICS = None
if _bp.exists():
    with open(_bp) as f:
        BASELINE_METRICS = json.load(f)
    print("Baseline (notebook 03) 결과:")
    print(f"  Type Accuracy   : {BASELINE_METRICS['type_accuracy']*100:.1f}%")
    print(f"  JSON Parse Rate : {BASELINE_METRICS['json_parse_rate']*100:.1f}%")
    print(f"  Severity Acc    : {BASELINE_METRICS['severity_accuracy']*100:.1f}%")
else:
    print("baseline_metrics.json 없음 — 04_evaluation.ipynb를 먼저 실행하세요.")
    print("실험 비교 시 baseline 열이 비어있게 됩니다.")

## 1. 데이터 증강 파이프라인

### 왜 증강인가?
NEU train set은 클래스당 210장으로 적다.  
이미지 레벨 증강으로 다양성을 높이면 과적합을 줄일 수 있다.

| 변환 | 확률 | 의도 |
|------|------|------|
| RandomRotate90 | 0.5 | 방향 무관 불량 특성 |
| HorizontalFlip | 0.5 | 공간 대칭성 |
| VerticalFlip | 0.3 | 공간 대칭성 |
| RandomBrightnessContrast | 0.5 | 조명 조건 변화 |
| GaussNoise | 0.3 | 센서 노이즈 시뮬레이션 |
| Blur (blur_limit=3) | 0.2 | 포커스 변동 |

In [ ]:
TRAIN_TRANSFORM = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Blur(blur_limit=3, p=0.2),
])

# 증강 결과 미리보기
with open(DATA_PROCESSED / "train.json", encoding="utf-8") as f:
    _sample = json.load(f)
_p = Path(_sample[0]["image"])
_img_path = (ROOT / _p) if not _p.is_absolute() else _p
_orig = np.array(Image.open(_img_path).convert("RGB"))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(_orig, cmap="gray")
axes[0, 0].set_title("원본")
axes[0, 0].axis("off")
for i, ax in enumerate(axes.flat[1:]):
    aug = TRAIN_TRANSFORM(image=_orig)["image"]
    ax.imshow(aug, cmap="gray")
    ax.set_title(f"증강 {i+1}")
    ax.axis("off")
plt.suptitle("albumentations 데이터 증강 예시", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "augmentation_preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장: data/results/augmentation_preview.png")

In [ ]:
def resolve_image(path: str) -> Path:
    p = Path(path)
    return (ROOT / p) if not p.is_absolute() else p


class DefectVQADataset(Dataset):
    """albumentations transform 지원 VQA 데이터셋"""

    def __init__(self, json_path, processor, max_length=512, transform=None):
        with open(json_path, encoding="utf-8") as f:
            self.data = json.load(f)
        self.processor   = processor
        self.max_length  = max_length
        self.transform   = transform
        self._asst_ids   = torch.tensor(
            processor.tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        record = self.data[idx]
        img = Image.open(resolve_image(record["image"])).convert("RGB")

        if self.transform is not None:
            img = Image.fromarray(self.transform(image=np.array(img))["image"])

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": record["conversations"][0]["content"]},
            ]},
            {"role": "assistant", "content": record["conversations"][1]["content"]},
        ]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        inputs = self.processor(
            text=[text], images=[img], return_tensors="pt",
            padding="max_length", max_length=self.max_length, truncation=True,
        )
        input_ids      = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        pv = inputs.get("pixel_values")
        pixel_values = pv.squeeze(0) if pv is not None else None

        igthw = inputs.get("image_grid_thw")
        image_grid_thw = igthw.squeeze(0) if igthw is not None else None

        labels = input_ids.clone()
        n = len(self._asst_ids)
        found = False
        for i in range(len(labels) - n):
            if (input_ids[i: i + n] == self._asst_ids).all():
                labels[: i + n] = -100
                found = True
                break
        if not found:
            labels[:] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "image_grid_thw": image_grid_thw,
            "labels": labels,
        }


## 2. 실험 설정

각 실험은 독립된 체크포인트 디렉토리에 저장된다.  
이미 결과 파일이 있으면 재실행을 생략한다.

In [ ]:
EXPERIMENT_CONFIGS = {
    "A": {
        "name": "rank32",
        "description": "LoRA rank 16 → 32  (표현력 향상)",
        "lora_rank": 32, "lora_alpha": 64,
        "augmentation": False, "label_smoothing": 0.0,
        "early_stopping": False, "es_patience": 2,
        "lr": 2e-4, "epochs": 3, "grad_accum": 8,
    },
    "B": {
        "name": "augmented",
        "description": "이미지 증강 추가  (일반화 향상)",
        "lora_rank": 16, "lora_alpha": 32,
        "augmentation": True, "label_smoothing": 0.0,
        "early_stopping": False, "es_patience": 2,
        "lr": 2e-4, "epochs": 3, "grad_accum": 8,
    },
    "C": {
        "name": "improved_train",
        "description": "레이블 스무딩 0.1 + 얼리스토핑  (과적합 방지)",
        "lora_rank": 16, "lora_alpha": 32,
        "augmentation": False, "label_smoothing": 0.1,
        "early_stopping": True, "es_patience": 2,
        "lr": 2e-4, "epochs": 5, "grad_accum": 8,
    },
    "D": {
        "name": "best_combo",
        "description": "rank32 + 증강 + 레이블스무딩 + 얼리스토핑  (종합)",
        "lora_rank": 32, "lora_alpha": 64,
        "augmentation": True, "label_smoothing": 0.1,
        "early_stopping": True, "es_patience": 2,
        "lr": 2e-4, "epochs": 5, "grad_accum": 8,
    },
}

print("실험 목록:")
print(f"  {'ID':<4} {'이름':<18} {'rank':<6} {'aug':<5} {'smooth':<8} {'early_stop'}")
print("-" * 55)
for k, v in EXPERIMENT_CONFIGS.items():
    print(f"  [{k}]  {v['name']:<18} {v['lora_rank']:<6} "
          f"{'✓' if v['augmentation'] else '✗':<5} "
          f"{v['label_smoothing']:<8} "
          f"{'✓' if v['early_stopping'] else '✗'}")

In [ ]:
# ──────────────────────────────────────────────────────────
#  실행할 실험을 선택하세요
#  ["A", "B", "C", "D"]  → 전부 실행 (~11시간)
#  ["D"]                 → best_combo만 (~3시간)
#  []                    → 기존 결과만 비교 (학습 없음)
# ──────────────────────────────────────────────────────────
EXPERIMENTS_TO_RUN = ["D"]

print(f"실행 예정: {EXPERIMENTS_TO_RUN}")
for eid in EXPERIMENTS_TO_RUN:
    print(f"  [{eid}] {EXPERIMENT_CONFIGS[eid]['description']}")

## 3. 학습 유틸리티

In [ ]:
def collate_fn(batch):
    result = {}
    for k in batch[0].keys():
        vals = [b[k] for b in batch if b[k] is not None]
        if vals:
            result[k] = torch.stack(vals)
    return result


def compute_loss(model, batch, smoothing=0.0):
    """label smoothing 지원. smoothing=0이면 모델 내부 loss 사용."""
    if smoothing == 0.0:
        return model(**batch).loss

    labels = batch.pop("labels")
    outputs = model(**batch)
    batch["labels"] = labels  # 복원

    shift_logits = outputs.logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fn = nn.CrossEntropyLoss(label_smoothing=smoothing, ignore_index=-100)
    return loss_fn(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
    )


class EarlyStopping:
    """val_loss가 patience 에포크 동안 개선 없으면 중단 신호 발생"""

    def __init__(self, patience=2, delta=1e-4):
        self.patience   = patience
        self.delta      = delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.should_stop = False

    def step(self, val_loss) -> bool:
        """개선 시 True 반환 (best 저장 신호)"""
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter   = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False


def load_model(cfg: dict):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=cfg["lora_rank"], lora_alpha=cfg["lora_alpha"],
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()
    return model, processor

In [ ]:
def run_experiment(exp_id: str, cfg: dict) -> dict:
    save_dir    = EXP_CKPT_DIR / cfg["name"]
    result_path = RESULTS_DIR / f"exp_{cfg['name']}_metrics.json"

    if result_path.exists():
        print(f"[{exp_id}] {cfg['name']} — 기존 결과 로드 (재실행 생략)")
        with open(result_path) as f:
            return json.load(f)

    print(f"\n{'='*56}")
    print(f"[{exp_id}] {cfg['description']}")
    print(f"{'='*56}")

    model, processor = load_model(cfg)
    device = next(model.parameters()).device
    save_dir.mkdir(parents=True, exist_ok=True)

    aug = TRAIN_TRANSFORM if cfg["augmentation"] else None
    train_ds = DefectVQADataset(DATA_PROCESSED / "train.json", processor, transform=aug)
    val_ds   = DefectVQADataset(DATA_PROCESSED / "val.json",   processor, transform=None)
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)

    GA           = cfg["grad_accum"]
    total_steps  = len(train_loader) * cfg["epochs"] // GA
    warmup_steps = int(total_steps * 0.1)
    optimizer    = bnb.optim.AdamW8bit(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg["lr"], weight_decay=0.01,
    )
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    es           = EarlyStopping(patience=cfg["es_patience"])
    smoothing    = cfg["label_smoothing"]

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    global_step   = 0

    def _optim_step():
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0
        )
        optimizer.step(); scheduler.step(); optimizer.zero_grad()

    t0 = time.time()
    for epoch in range(cfg["epochs"]):
        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"[{exp_id}] Ep{epoch+1} train")
        for step, batch in enumerate(pbar):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss  = compute_loss(model, batch, smoothing) / GA
            loss.backward()
            epoch_loss += loss.item() * GA
            if (step + 1) % GA == 0:
                _optim_step(); global_step += 1
            pbar.set_postfix(loss=f"{loss.item()*GA:.4f}")
        if len(train_loader) % GA != 0:
            _optim_step(); global_step += 1

        avg_train = epoch_loss / len(train_loader)
        train_losses.append(avg_train)

        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"[{exp_id}] Ep{epoch+1} val"):
                batch = {k: v.to(device) for k, v in batch.items()}
                val_sum += model(**batch).loss.item()
        avg_val = val_sum / len(val_loader)
        val_losses.append(avg_val)
        print(f"\nEpoch {epoch+1}: train={avg_train:.4f}  val={avg_val:.4f}")

        # early_stopping 사용 시 EarlyStopping.step()으로 판정, 아니면 단순 비교
        improved = es.step(avg_val) if cfg["early_stopping"] else (avg_val < best_val_loss)
        if improved:
            best_val_loss = avg_val
            model.save_pretrained(save_dir)
            processor.save_pretrained(save_dir)
            print(f"  ✓ Best 저장 (val={best_val_loss:.4f})")

        if cfg["early_stopping"] and es.should_stop:
            print(f"  얼리스토핑 (patience={cfg['es_patience']} 소진)")
            break

    elapsed = (time.time() - t0) / 60
    print(f"\n[{exp_id}] 완료: {elapsed:.1f}분  best_val={best_val_loss:.4f}")

    # 학습 곡선
    fig, ax = plt.subplots(figsize=(7, 4))
    xs = range(1, len(train_losses) + 1)
    ax.plot(xs, train_losses, "o-", label="Train", color="#4C72B0")
    ax.plot(xs, val_losses,   "s-", label="Val",   color="#C44E52")
    ax.set_title(f"[{exp_id}] {cfg['name']} — 학습 곡선")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"exp_{cfg['name']}_curve.png", dpi=120, bbox_inches="tight")
    plt.show()

    metrics = {
        "exp_id": exp_id, "name": cfg["name"],
        "description": cfg["description"],
        "best_val_loss": round(best_val_loss, 4),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "elapsed_min": round(elapsed, 1),
        "lora_rank": cfg["lora_rank"],
        "augmentation": cfg["augmentation"],
        "label_smoothing": cfg["label_smoothing"],
        "early_stopping": cfg["early_stopping"],
    }
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    del model
    torch.cuda.empty_cache()
    return metrics

## 4. 실험 실행

> 이미 `data/results/exp_<name>_metrics.json`가 있으면 재실행을 생략합니다.

In [ ]:
exp_results = {}

for exp_id in EXPERIMENTS_TO_RUN:
    if exp_id not in EXPERIMENT_CONFIGS:
        print(f"알 수 없는 실험 ID: {exp_id}  (A/B/C/D 중 선택)")
        continue
    exp_results[exp_id] = run_experiment(exp_id, EXPERIMENT_CONFIGS[exp_id])

print(f"\n완료: {list(exp_results.keys())}")

## 5. 결과 비교 & 시각화

val_loss 기준 비교 + 학습 곡선 멀티 플롯.  
정량적 Type Accuracy 비교는 각 체크포인트로 `04_evaluation.ipynb` 스타일 평가를 별도 실행하세요.

In [ ]:
# 완료된 실험 결과 수집 (기존 저장 파일 포함)
all_results = {}

for exp_id, cfg in EXPERIMENT_CONFIGS.items():
    rp = RESULTS_DIR / f"exp_{cfg['name']}_metrics.json"
    if rp.exists():
        with open(rp) as f:
            all_results[exp_id] = json.load(f)
        m = all_results[exp_id]
        print(f"[{exp_id}] {cfg['name']:<20} best_val_loss={m['best_val_loss']:.4f}  ({m['elapsed_min']:.0f}분)")

if not all_results:
    print("비교할 결과가 없습니다. 먼저 실험을 실행하세요.")

In [ ]:
if not all_results:
    print("실험 결과 없음 — 비교 생략")
else:
    ids    = list(all_results.keys())
    names  = [all_results[k]["name"] for k in ids]
    losses = [all_results[k]["best_val_loss"] for k in ids]

    # baseline 추가
    if BASELINE_METRICS and "baseline" not in names:
        ids.insert(0, "baseline")
        names.insert(0, "baseline (rank16)")
        # val_loss는 baseline에 없으므로 None 처리
        losses.insert(0, None)

    # val_loss 비교 (None 제외)
    plot_ids = [i for i, v in zip(ids, losses) if v is not None]
    plot_names  = [names[ids.index(i)] for i in plot_ids]
    plot_losses = [all_results[i]["best_val_loss"] for i in plot_ids]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 왼쪽: val_loss 막대
    colors = ["#1565C0" if i == np.argmin(plot_losses) else "#90CAF9"
              for i in range(len(plot_ids))]
    bars = axes[0].bar(plot_names, plot_losses, color=colors)
    axes[0].set_ylabel("Best Val Loss (낮을수록 좋음)")
    axes[0].set_title("실험별 Best Val Loss 비교", fontsize=13)
    _ymax = max(plot_losses) * 1.15
    axes[0].set_ylim(0, _ymax)
    for bar, val in zip(bars, plot_losses):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + _ymax * 0.01,
                     f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
    axes[0].tick_params(axis="x", rotation=15)

    # 오른쪽: 학습 곡선 오버레이
    colors_line = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    for i, eid in enumerate(plot_ids):
        m = all_results[eid]
        xs = range(1, len(m["val_losses"]) + 1)
        axes[1].plot(xs, m["val_losses"], "o-",
                     label=m["name"], color=colors_line[i % len(colors_line)])
    axes[1].set_title("실험별 Val Loss 곡선", fontsize=13)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val Loss")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle("실험 비교", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "exp_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: data/results/exp_comparison.png")

    # 요약 테이블
    rows = []
    for eid in plot_ids:
        m = all_results[eid]
        rows.append({
            "ID": eid, "이름": m["name"],
            "rank": m["lora_rank"],
            "aug": "✓" if m["augmentation"] else "✗",
            "smooth": m["label_smoothing"],
            "early_stop": "✓" if m["early_stopping"] else "✗",
            "best_val_loss": m["best_val_loss"],
            "학습(분)": m["elapsed_min"],
        })
    df_cmp = pd.DataFrame(rows).set_index("ID")
    print("\n" + df_cmp.to_string())

In [ ]:
# Best 실험 → models/checkpoints/best_exp 으로 복사
if all_results:
    best_id   = min(all_results, key=lambda k: all_results[k]["best_val_loss"])
    best_name = all_results[best_id]["name"]
    best_loss = all_results[best_id]["best_val_loss"]
    print(f"최고 실험: [{best_id}] {best_name}  (val_loss={best_loss:.4f})")

    src = EXP_CKPT_DIR / best_name
    dst = ROOT / "models" / "checkpoints" / "best_exp"
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"체크포인트 복사: .../{best_name} → .../best_exp")
        print("demo.py 또는 app/main.py의 LORA_PATH를 best_exp로 변경하면 새 모델 사용 가능합니다.")
    else:
        print(f"체크포인트 없음: {src}")

    summary = {
        "best_experiment": best_id,
        "best_name": best_name,
        "best_val_loss": best_loss,
        "experiments": {
            k: {"name": v["name"], "best_val_loss": v["best_val_loss"]}
            for k, v in all_results.items()
        }
    }
    with open(RESULTS_DIR / "exp_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print("저장: data/results/exp_summary.json")
    print("\n다음 단계: 04_evaluation.ipynb의 MODEL_CKPT를 best_exp로 변경해 Type Accuracy 재평가")
else:
    print("완료된 실험 없음")

## 6. Best Experiment — 테스트셋 전체 평가

`best_exp` 체크포인트를 Test Set으로 추론하여  
**Zero-shot → rank16 파인튜닝 → Best Experiment** 3단계 비교표를 생성한다.

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

with open(DATA_PROCESSED / "test.json", encoding="utf-8") as f:
    test_data = json.load(f)

BEST_EXP_CKPT = ROOT / "models" / "checkpoints" / "best_exp"
EVAL_RESULT_PATH  = RESULTS_DIR / "exp_best_eval_metrics.json"
EVAL_CSV_PATH     = RESULTS_DIR / "exp_best_eval_results.csv"
EVAL_PARTIAL_PATH = RESULTS_DIR / "exp_best_eval_partial.csv"

# 이미 평가 완료된 경우 재실행 생략
_skip_eval = EVAL_RESULT_PATH.exists()

if not BEST_EXP_CKPT.exists():
    print(f"best_exp 체크포인트 없음: {BEST_EXP_CKPT}")
    print("먼저 위 셀에서 실험을 실행하거나, 수동으로 체크포인트를 복사하세요.")
    _skip_eval = True
elif _skip_eval:
    print(f"평가 결과 이미 존재 — 로드합니다: {EVAL_RESULT_PATH}")
else:
    print(f"Best Exp 체크포인트: {BEST_EXP_CKPT}")
    print(f"Test set: {len(test_data)}개")


In [ ]:
if not _skip_eval:
    from peft import PeftModel

    MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    _base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb_cfg,
        device_map="auto", torch_dtype=torch.bfloat16,
    )
    _proc = AutoProcessor.from_pretrained(BEST_EXP_CKPT)
    eval_model = PeftModel.from_pretrained(_base, BEST_EXP_CKPT)
    eval_model.eval()
    print("best_exp 모델 로드 완료")

In [ ]:
if not _skip_eval:
    def _infer_exp(image_path: str) -> tuple[str, float]:
        img = Image.open(resolve_image(image_path)).convert("RGB")
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": INFERENCE_PROMPT},
            ]},
        ]
        text = _proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = _proc(text=[text], images=[img], return_tensors="pt", padding=True).to(eval_model.device)
        t0 = time.time()
        with torch.no_grad():
            out_ids = eval_model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
                temperature=None, top_p=None,
            )
        elapsed = time.time() - t0
        generated = out_ids[:, inputs["input_ids"].shape[1]:]
        return _proc.batch_decode(generated, skip_special_tokens=True)[0].strip(), elapsed

    def _parse(raw: str) -> dict | None:
        raw = re.sub(r"```json\s*", "", raw)
        raw = re.sub(r"```\s*", "", raw)
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            return None

    # 체크포인트 지원 (중단 재개)
    done_ids: set[str] = set()
    exp_results_rows: list[dict] = []
    if EVAL_PARTIAL_PATH.exists():
        _prev = pd.read_csv(EVAL_PARTIAL_PATH)
        exp_results_rows = _prev.to_dict("records")
        done_ids = set(_prev["id"])
        print(f"체크포인트 로드: {len(done_ids)}개 완료")

    remaining = [r for r in test_data if r["id"] not in done_ids]
    print(f"남은 샘플: {len(remaining)}개")

    CHECKPOINT_EVERY = 50
    for i, record in enumerate(tqdm(remaining, desc="best_exp 추론")):
        gt = json.loads(record["conversations"][1]["content"])
        raw, elapsed = _infer_exp(record["image"])
        parsed = _parse(raw)
        pred_type, pred_severity = None, None
        if parsed:
            pred_type = parsed.get("type", "").strip().lower()
            pred_severity = parsed.get("severity", "").strip().lower()
            if pred_type not in DEFECT_CLASSES:
                pred_type = None
        exp_results_rows.append({
            "id": record["id"],
            "image": record["image"],
            "gt_type": gt["type"],
            "gt_severity": gt["severity"],
            "pred_type": pred_type,
            "pred_severity": pred_severity,
            "json_ok": parsed is not None,
            "elapsed": elapsed,
        })
        if (i + 1) % CHECKPOINT_EVERY == 0:
            pd.DataFrame(exp_results_rows).to_csv(EVAL_PARTIAL_PATH, index=False)

    exp_df = pd.DataFrame(exp_results_rows)
    exp_json_rate = exp_df["json_ok"].mean()
    exp_type_acc  = (exp_df["pred_type"] == exp_df["gt_type"]).mean()
    exp_sev_acc   = (exp_df["pred_severity"] == exp_df["gt_severity"]).mean()
    exp_avg_time  = exp_df["elapsed"].mean()

    print("=" * 50)
    print(f"  Best Exp ({best_name}) 결과")
    print("=" * 50)
    print(f"  JSON Parse Rate  : {exp_json_rate*100:.1f}%")
    print(f"  Type Accuracy    : {exp_type_acc*100:.1f}%")
    print(f"  Severity Accuracy: {exp_sev_acc*100:.1f}%")
    print(f"  Avg Latency      : {exp_avg_time:.2f}s/image")

    exp_eval_metrics = {
        "model": f"Qwen2.5-VL-7B-Instruct (QLoRA {best_name})",
        "exp_id": best_id,
        "exp_name": best_name,
        "json_parse_rate": round(float(exp_json_rate), 4),
        "type_accuracy": round(float(exp_type_acc), 4),
        "severity_accuracy": round(float(exp_sev_acc), 4),
        "avg_latency_sec": round(float(exp_avg_time), 3),
        "n_test": len(test_data),
    }
    with open(EVAL_RESULT_PATH, "w", encoding="utf-8") as f:
        json.dump(exp_eval_metrics, f, indent=2, ensure_ascii=False)
    exp_df.to_csv(EVAL_CSV_PATH, index=False)
    EVAL_PARTIAL_PATH.unlink(missing_ok=True)

    del eval_model, _base
    torch.cuda.empty_cache()

In [ ]:
# 결과 로드 (이미 존재하거나 방금 평가한 경우)
if EVAL_RESULT_PATH.exists():
    with open(EVAL_RESULT_PATH, encoding="utf-8") as f:
        exp_eval_metrics = json.load(f)
    exp_df = pd.read_csv(EVAL_CSV_PATH)
    print(f"[{exp_eval_metrics['exp_id']}] {exp_eval_metrics['exp_name']}")
    print(f"  Type Accuracy    : {exp_eval_metrics['type_accuracy']*100:.1f}%")
    print(f"  JSON Parse Rate  : {exp_eval_metrics['json_parse_rate']*100:.1f}%")
    print(f"  Severity Accuracy: {exp_eval_metrics['severity_accuracy']*100:.1f}%")
else:
    print("평가 결과 없음 — 위 셀에서 best_exp 체크포인트가 있는지 확인하세요.")
    exp_eval_metrics = None

In [ ]:
# 3단계 비교: Zero-shot → rank16 파인튜닝 → Best Experiment
if exp_eval_metrics is None:
    print("비교 데이터 없음")
else:
    _baseline_path  = RESULTS_DIR / "baseline_metrics.json"
    _finetuned_path = RESULTS_DIR / "finetuned_metrics.json"

    stages = []
    if _baseline_path.exists():
        with open(_baseline_path) as f:
            stages.append(("Zero-shot", json.load(f)))
    if _finetuned_path.exists():
        with open(_finetuned_path) as f:
            stages.append(("rank16 QLoRA", json.load(f)))
    stages.append((f"Best Exp\n({exp_eval_metrics['exp_name']})", exp_eval_metrics))

    metric_keys   = ["json_parse_rate", "type_accuracy", "severity_accuracy"]
    metric_labels = ["JSON Parse Rate", "Type Accuracy", "Severity Accuracy"]
    colors_stage  = ["#90CAF9", "#42A5F5", "#0D47A1"]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # 왼쪽: 3단계 그룹 막대
    x = np.arange(len(metric_labels))
    w = 0.25
    for i, (stage_name, m) in enumerate(stages):
        vals = [m[k] * 100 for k in metric_keys]
        bars = axes[0].bar(x + (i - 1) * w, vals, w,
                           label=stage_name, color=colors_stage[i])
        for bar, val in zip(bars, vals):
            axes[0].text(bar.get_x() + bar.get_width() / 2,
                         bar.get_height() + 0.8,
                         f"{val:.1f}", ha="center", fontsize=8, fontweight="bold")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metric_labels, rotation=10)
    axes[0].set_ylabel("(%)")
    axes[0].set_ylim(0, 115)
    axes[0].set_title("3단계 성능 비교", fontsize=13)
    axes[0].legend(loc="lower right")

    # 오른쪽: Type Accuracy 단계별 추이 (꺾은선)
    stage_names_short = [s[0].replace("\n", " ") for s in stages]
    ta_vals = [m["type_accuracy"] * 100 for _, m in stages]
    axes[1].plot(range(len(stages)), ta_vals, "o-", color="#0D47A1",
                 markersize=10, linewidth=2.5)
    for i, (name, val) in enumerate(zip(stage_names_short, ta_vals)):
        axes[1].annotate(f"{val:.1f}%", (i, val),
                         textcoords="offset points", xytext=(0, 10),
                         ha="center", fontsize=11, fontweight="bold")
    axes[1].set_xticks(range(len(stages)))
    axes[1].set_xticklabels(stage_names_short, rotation=10)
    axes[1].set_ylabel("Type Accuracy (%)")
    axes[1].set_ylim(max(0, min(ta_vals) - 10), 105)
    axes[1].set_title("Type Accuracy 개선 추이", fontsize=13)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle("Zero-shot → rank16 → Best Exp 성능 비교", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "three_stage_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: data/results/three_stage_comparison.png")

    # 요약 테이블 출력
    rows = []
    for stage_name, m in stages:
        rows.append({
            "단계": stage_name.replace("\n", " "),
            "Type Acc (%)": f"{m['type_accuracy']*100:.1f}",
            "JSON Parse (%)": f"{m['json_parse_rate']*100:.1f}",
            "Severity Acc (%)": f"{m['severity_accuracy']*100:.1f}",
        })
    df_summary = pd.DataFrame(rows).set_index("단계")
    print("\n" + df_summary.to_string())

In [ ]:
# Best Exp 혼동 행렬
if exp_eval_metrics is not None and EVAL_CSV_PATH.exists():
    valid_df = exp_df.dropna(subset=["pred_type"])

    print(f"Best Exp ({exp_eval_metrics['exp_name']}) — 클래스별 성능:")
    print(classification_report(
        valid_df["gt_type"], valid_df["pred_type"],
        labels=DEFECT_CLASSES, zero_division=0,
    ))

    cm = confusion_matrix(valid_df["gt_type"], valid_df["pred_type"], labels=DEFECT_CLASSES)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=DEFECT_CLASSES, yticklabels=DEFECT_CLASSES, ax=ax)
    ax.set_xlabel("예측", fontsize=12)
    ax.set_ylabel("정답", fontsize=12)
    ax.set_title(f"Best Exp ({exp_eval_metrics['exp_name']}) 혼동 행렬", fontsize=13)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "exp_best_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("저장: data/results/exp_best_confusion_matrix.png")